# Health Index Improved Pipeline

**Purpose**: Modular, scalable telemetry-based health index modeling with experiment tracking.

**Key improvements over original**:
- Optuna hyperparameter optimization
- MLflow experiment tracking (JSON-based)
- Modular function-based design
- Multi-window inference support
- Multi-component execution
- Proper artifact management
- SOLID principles compliance

**Pipeline stages**:
1. Data loading
2. Outlier cleaning
3. Cycle-based interpolation
4. Labeling
5. Feature preparation & scaling
6. Model training with optimization
7. Multi-window inference
8. Health index computation
9. Artifact persistence

## 1. Imports and Configuration

In [1]:
# Standard libraries
import os
import json
import warnings
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Any

# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning
import tensorflow as tf
from tensorflow.keras import layers, Model, mixed_precision
from sklearn.preprocessing import RobustScaler, OneHotEncoder

# Optimization and tracking
import optuna
import mlflow
from mlflow.tracking import MlflowClient

# Utilities
import joblib
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

### GPU Configuration

In [2]:
def configure_gpu():
    """Configure GPU settings for optimal performance."""
    print("TensorFlow version:", tf.__version__)
    print("GPU Available:", tf.config.list_physical_devices('GPU'))
    print("Built with CUDA:", tf.test.is_built_with_cuda())
    
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            print(f"✓ GPU memory growth enabled for {len(gpus)} GPU(s)")
            
            policy = mixed_precision.Policy('mixed_float16')
            mixed_precision.set_global_policy(policy)
            print(f"✓ Mixed precision enabled: {policy.name}")
        except RuntimeError as e:
            print(f"GPU configuration error: {e}")
    else:
        print("⚠  No GPU detected - training will use CPU")

configure_gpu()

TensorFlow version: 2.21.0
GPU Available: []
Built with CUDA: False
⚠  No GPU detected - training will use CPU


### Pipeline Configuration

In [ ]:
class PipelineConfig:
    """Central configuration for the health index pipeline."""
    
    # Column names
    UNIT_COL = "Unit"
    TIME_COL = "Fecha"
    CAT_COLS = ["EstadoMaquina", "EstadoCarga"]
    
    # Data cleaning
    FREQ = "1min"
    GAP_THRESHOLD = pd.Timedelta("10min")
    MIN_DURATION = pd.Timedelta("4h")
    MIN_COVERAGE = 0.75
    INTERP_LIMIT = 10
    
    # Labeling
    PERCENTILE_LOW = 0.05
    PERCENTILE_HIGH = 0.95
    ANOMALY_THRESHOLD = 1.8  # ratio threshold for anomaly labeling
    
    # Model training
    WINDOW_SIZE = 60  # 1 hour at 1-minute frequency
    DROPOUT_RATE = 0.2
    EARLY_STOPPING_PATIENCE = 5
    
    # Testing
    WEEKS_TO_TEST = 12
    
    # Health index
    HI_ALPHA = 1.0  # decay factor for health index calculation
    HI_AGG_METHOD = "mean"  # aggregation method: mean, median
    
    # Paths
    CLIENT = "cda"
    BASE_DATA_PATH = Path("../data")
    BASE_MODEL_PATH = Path("../models")
    
    @classmethod
    def get_signal_margins(cls) -> Dict[str, Tuple[float, float]]:
        """Define valid ranges for each signal."""
        return {
            # General
            'GPSLat': (-30.4, -30.1),
            'GPSLon': (-71.3, -70.9),
            'GPSElevation': (400, 2000),
            'GroundSpd': (0, 80),
            'EngSpd': (0, 2500),
            # Engine
            "EngCoolTemp": (30, 120),
            "RAftrclrTemp": (10, 100),
            "EngOilPres": (150, 700),
            "EngOilFltr": (1, 50),
            "CnkcasePres": (-1.5, 1.5),
            "RtLtExhTemp": (-10, 10),
            "RtExhTemp": (150, 750),
            "LtExhTemp": (150, 750),
            # Transmission
            "DiffLubePres": (0, 800),
            "DiffTemp": (0, 150),
            "TrnLubeTemp": (-5, 120),
            "TCOutTemp": (30, 180),
            # Brakes
            "RtRBrkTemp": (20, 200),
            "RtFBrkTemp": (20, 200),
            "LtRBrkTemp": (20, 200),
            "LtFBrkTemp": (20, 200),
            # Direction
            'StrgOilTemp': (-10, 150),
        }

config = PipelineConfig()
print("✓ Configuration loaded")

✓ Configuration loaded


## 2. Data Loading

In [4]:
def load_telemetry_data(
    client: str = config.CLIENT,
    data_type: str = "silver"
) -> pd.DataFrame:
    """
    Load telemetry data from parquet files.
    
    Args:
        client: Client identifier (e.g., 'cda')
        data_type: Data layer (silver, golden)
    
    Returns:
        DataFrame with telemetry data
    """
    file_path = config.BASE_DATA_PATH / f"telemetry/{data_type}/{client}/Telemetry_Wide_With_States"
    
    if not Path(file_path).exists():
        # Try with .parquet extension
        file_path = Path(str(file_path) + ".parquet")
    
    print(f"Loading data from: {file_path}")
    df = pd.read_parquet(file_path)
    
    # Sort and remove duplicates
    df.sort_values([config.UNIT_COL, config.TIME_COL], inplace=True)
    df.drop_duplicates(subset=[config.UNIT_COL, config.TIME_COL], keep='first', inplace=True)
    
    # Drop problematic columns if they exist
    drop_cols = ['Payload', 'EngOilFltr', 'AirFltr']
    existing_drop_cols = [col for col in drop_cols if col in df.columns]
    if existing_drop_cols:
        df.drop(columns=existing_drop_cols, inplace=True)
    
    print(f"✓ Loaded {len(df):,} rows ({len(df)/1e6:.2f}M)")
    print(f"✓ Units: {df[config.UNIT_COL].nunique()}")
    print(f"✓ Time range: {df[config.TIME_COL].min()} to {df[config.TIME_COL].max()}")
    
    return df


def load_component_mapping(client: str = config.CLIENT) -> Dict:
    """
    Load component-to-signals mapping configuration.
    
    Args:
        client: Client identifier
    
    Returns:
        Dictionary with component configuration
    """
    mapping_path = config.BASE_DATA_PATH / "telemetry/component_signals_mapping.json"
    
    with open(mapping_path, 'r') as f:
        mapping = json.load(f)
    
    print(f"✓ Loaded component mapping with {len(mapping['components'])} components")
    return mapping

## 3. Data Cleaning

In [5]:
def clean_outliers(
    df: pd.DataFrame,
    margins: Dict[str, Tuple[float, float]]
) -> pd.DataFrame:
    """
    Replace values outside valid ranges with NaN.
    
    Args:
        df: Input DataFrame
        margins: Dictionary mapping column names to (min, max) tuples
    
    Returns:
        Cleaned DataFrame
    """
    df_clean = df.copy()
    
    for col, (lower, upper) in margins.items():
        if col in df_clean.columns:
            original_valid = df_clean[col].notna().sum()
            df_clean[col] = df_clean[col].where(
                (df_clean[col] >= lower) & (df_clean[col] <= upper),
                other=pd.NA
            )
            new_valid = df_clean[col].notna().sum()
            if original_valid != new_valid:
                pct_removed = (original_valid - new_valid) / original_valid * 100
                print(f"  {col}: removed {original_valid - new_valid:,} outliers ({pct_removed:.1f}%)")
    
    return df_clean


def drop_incomplete_rows(
    df: pd.DataFrame,
    signal_cols: List[str],
    threshold_ratio: float = 0.5
) -> pd.DataFrame:
    """
    Drop rows with too many missing signals.
    
    Args:
        df: Input DataFrame
        signal_cols: List of signal columns to check
        threshold_ratio: Minimum ratio of non-null signals required
    
    Returns:
        Filtered DataFrame
    """
    existing_cols = [col for col in signal_cols if col in df.columns]
    min_valid = int(len(existing_cols) * threshold_ratio)
    
    initial_rows = len(df)
    df_filtered = df.dropna(subset=existing_cols, thresh=min_valid).copy()
    
    # Fill categorical columns
    df_filtered.fillna({col: 'ND' for col in config.CAT_COLS if col in df_filtered.columns}, inplace=True)
    df_filtered.reset_index(drop=True, inplace=True)
    
    removed = initial_rows - len(df_filtered)
    print(f"✓ Dropped {removed:,} incomplete rows ({removed/initial_rows*100:.1f}%)")
    print(f"✓ Remaining: {len(df_filtered):,} rows")
    
    return df_filtered

## 4. Cycle Creation and Interpolation

In [6]:
def create_cycles(df: pd.DataFrame) -> pd.DataFrame:
    """
    Identify operational cycles based on time gaps.
    
    A new cycle starts when:
    - There's a gap larger than GAP_THRESHOLD between consecutive records
    - OR it's the first record for a unit
    
    Args:
        df: Input DataFrame with Unit and Fecha columns
    
    Returns:
        DataFrame with cycle_id column added
    """
    df_cycles = df.copy()
    
    # Calculate time differences within each unit
    dt = df_cycles.groupby(config.UNIT_COL)[config.TIME_COL].diff()
    
    # Mark cycle boundaries
    new_cycle = dt.isna() | (dt > config.GAP_THRESHOLD)
    df_cycles["cycle_id"] = new_cycle.groupby(df_cycles[config.UNIT_COL]).cumsum().astype("int64")
    
    n_cycles = df_cycles.groupby(config.UNIT_COL)["cycle_id"].nunique().sum()
    print(f"✓ Created {n_cycles:,} cycles across {df_cycles[config.UNIT_COL].nunique()} units")
    
    return df_cycles


def is_valid_cycle(cycle_df: pd.DataFrame) -> bool:
    """
    Check if cycle meets minimum duration and coverage requirements.
    
    Args:
        cycle_df: DataFrame for a single cycle
    
    Returns:
        True if cycle is valid for analysis
    """
    start = cycle_df[config.TIME_COL].iloc[0]
    end = cycle_df[config.TIME_COL].iloc[-1]
    duration = end - start
    
    freq_td = pd.to_timedelta(config.FREQ)
    expected_n = int(round(duration / freq_td)) + 1
    coverage = len(cycle_df) / expected_n if expected_n > 0 else 0.0
    
    return (duration >= config.MIN_DURATION) and (coverage >= config.MIN_COVERAGE)


def interpolate_cycle(
    cycle_df: pd.DataFrame,
    num_cols: List[str]
) -> pd.DataFrame:
    """
    Interpolate missing values within a single cycle.
    
    Creates a complete time index at 1-minute frequency and interpolates
    numeric signals using time-based interpolation.
    
    Args:
        cycle_df: DataFrame for a single cycle
        num_cols: List of numeric column names to interpolate
    
    Returns:
        Interpolated DataFrame with complete time index
    """
    out = cycle_df.copy()
    out = out.sort_values(config.TIME_COL).reset_index(drop=True)
    
    # Track original timestamps
    original_timestamps = set(out[config.TIME_COL])
    
    # Build complete time index
    full_time_index = pd.date_range(
        start=out[config.TIME_COL].min(),
        end=out[config.TIME_COL].max(),
        freq=config.FREQ
    )
    
    # Reindex to create missing timestamps
    out = out.set_index(config.TIME_COL).reindex(full_time_index)
    out.index.name = config.TIME_COL
    
    # Reconstruct metadata columns
    if config.UNIT_COL in cycle_df.columns:
        out[config.UNIT_COL] = cycle_df[config.UNIT_COL].iloc[0]
    
    if "cycle_id" in cycle_df.columns:
        out["cycle_id"] = cycle_df["cycle_id"].iloc[0]
    
    # Forward-fill categorical columns
    for c in config.CAT_COLS:
        if c in cycle_df.columns:
            out[c] = out[c].ffill().bfill()
    
    # Filter to existing numeric columns
    valid_num_cols = [c for c in num_cols if c in out.columns]
    
    # Convert to numeric if needed
    for c in valid_num_cols:
        if out[c].dtype == "object" or pd.api.types.is_string_dtype(out[c]):
            out[c] = pd.to_numeric(out[c], errors="coerce")
    
    # Flag rows created by reindex
    out["created_by_reindex"] = (~out.index.isin(original_timestamps)).astype("int8")
    
    # Save missing positions before interpolation
    before_na = out[valid_num_cols].isna()
    
    # Interpolate numeric signals
    out[valid_num_cols] = out[valid_num_cols].interpolate(
        method="time",
        limit=config.INTERP_LIMIT,
        limit_area="inside",
    )
    
    # Flag rows where values were imputed
    out["imputed_any"] = (
        (before_na & ~out[valid_num_cols].isna()).any(axis=1)
    ).astype("int8")
    
    return out.reset_index()


def process_all_cycles(
    df: pd.DataFrame,
    num_cols: List[str]
) -> pd.DataFrame:
    """
    Process all cycles: filter valid ones and interpolate.
    
    Args:
        df: DataFrame with cycle_id column
        num_cols: List of numeric columns to interpolate
    
    Returns:
        DataFrame with interpolated cycles
    """
    cycles = []
    
    for (unit, cycle_id), cycle_df in tqdm(
        df.groupby([config.UNIT_COL, "cycle_id"], sort=False),
        desc="Processing cycles"
    ):
        if is_valid_cycle(cycle_df):
            interpolated = interpolate_cycle(cycle_df, num_cols)
            cycles.append(interpolated)
    
    result = pd.concat(cycles, ignore_index=True) if cycles else df.head(0).copy()
    
    print(f"✓ Processed {len(cycles):,} valid cycles")
    print(f"✓ Total rows after interpolation: {len(result):,}")
    
    return result

## 5. Data Labeling

In [7]:
def label_cycles_by_percentiles(
    df: pd.DataFrame,
    signal_cols: List[str]
) -> pd.DataFrame:
    """
    Label cycles as Normal or Anomalous based on percentile analysis.
    
    For each signal, calculates P5 and P95 across all data, then counts
    how many values per cycle fall outside this range. Cycles with high
    ratios of out-of-range values are labeled Anomalous.
    
    Args:
        df: DataFrame with cycle data
        signal_cols: List of signal columns to analyze
    
    Returns:
        DataFrame with 'Label' column added
    """
    df_labeled = df.copy()
    
    # Filter to relevant signal columns (exclude GPS and speed columns for labeling)
    label_cols = [
        col for col in signal_cols 
        if col in df.columns and 'GPS' not in col and 'Spd' not in col
    ]
    
    # Calculate percentiles
    percentile_dict = {
        col: df[col].quantile([config.PERCENTILE_LOW, config. PERCENTILE_HIGH])
        for col in label_cols
    }
    
    # Flag out-of-range values
    out_cols = []
    for col in percentile_dict.keys():
        out_col = f'{col}_out_range'
        df_labeled[out_col] = ~df_labeled[col].between(
            percentile_dict[col][config.PERCENTILE_LOW],
            percentile_dict[col][config.PERCENTILE_HIGH]
        )
        out_cols.append(out_col)
    
    # Aggregate at cycle level
    cycle_summary = df_labeled.groupby([config.UNIT_COL, 'cycle_id'])[out_cols].sum()
    cycle_total = df_labeled.groupby([config.UNIT_COL, 'cycle_id']).size().rename('total_rows')
    cycle_summary = cycle_summary.merge(cycle_total, left_index=True, right_index=True)
    
    cycle_summary['total_out_range'] = cycle_summary[out_cols].sum(axis=1)
    cycle_summary['total_ratio'] = cycle_summary['total_out_range'] / cycle_summary['total_rows']
    
    # Label based on threshold
    cycle_summary['Label'] = np.where(
        cycle_summary['total_ratio'] < config.ANOMALY_THRESHOLD,
        'Normal',
        'Anomalous'
    )
    
    # Merge labels back
    df_labeled = df_labeled.merge(
        cycle_summary[['Label']],
        left_on=[config.UNIT_COL, 'cycle_id'],
        right_index=True
    )
    
    # Clean up temporary columns
    df_labeled.drop(columns=out_cols, inplace=True)
    
    # Print statistics
    label_counts = cycle_summary['Label'].value_counts()
    print(f"✓ Labeling complete:")
    print(f"  Normal cycles: {label_counts.get('Normal', 0):,}")
    print(f"  Anomalous cycles: {label_counts.get('Anomalous', 0):,}")
    
    return df_labeled

## 6. Feature Preparation and Scaling

In [8]:
def fit_scalers_per_unit(
    df: pd.DataFrame,
    signal_cols: List[str],
    component: str,
    client: str = config.CLIENT
) -> Dict[str, Any]:
    """
    Fit separate robust scalers for each unit's signals.
    
    Also fits a shared OneHotEncoder for categorical variables.
    Scalers are trained only on Normal data.
    
    Args:
        df: Training DataFrame
        signal_cols: List of signal columns
        component: Component name
        client: Client identifier
    
    Returns:
        Dictionary with 'signal' and 'categorical' scalers
    """
    # Filter to Normal data only
    df_normal = df[df['Label'] == 'Normal'].copy() if 'Label' in df.columns else df.copy()
    
    # Fit categorical encoder
    cat_scaler = OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False)
    existing_cat_cols = [col for col in config.CAT_COLS if col in df.columns]
    cat_scaler.fit(df_normal[existing_cat_cols])
    
    # Fit signal scalers per unit
    signal_scalers = {}
    for unit in df_normal[config.UNIT_COL].unique():
        unit_data = df_normal[df_normal[config.UNIT_COL] == unit]
        scaler = RobustScaler()
        scaler.fit(unit_data[signal_cols])
        signal_scalers[unit] = scaler
    
    scalers = {
        'categorical': cat_scaler,
        'signal': signal_scalers
    }
    
    # Save scalers
    scaler_dir = config.BASE_MODEL_PATH / client / component / "scalers"
    scaler_dir.mkdir(parents=True, exist_ok=True)
    
    joblib.dump(cat_scaler, scaler_dir / "cat_scaler.pkl")
    for unit, scaler in signal_scalers.items():
        joblib.dump(scaler, scaler_dir / f"{unit}_signal_scaler.pkl")
    
    print(f"✓ Fitted scalers for {len(signal_scalers)} units")
    print(f"✓ Categorical features: {len(cat_scaler.get_feature_names_out())}")
    print(f"✓ Saved to: {scaler_dir}")
    
    return scalers


def load_scalers(
    df: pd.DataFrame,
    component: str,
    client: str = config.CLIENT
) -> Dict[str, Any]:
    """
    Load pre-fitted scalers from disk.
    
    Args:
        df: DataFrame (used to identify units)
        component: Component name
        client: Client identifier
    
    Returns:
        Dictionary with 'signal' and 'categorical' scalers
    """
    scaler_dir = config.BASE_MODEL_PATH / client / component / "scalers"
    
    cat_scaler = joblib.load(scaler_dir / "cat_scaler.pkl")
    
    signal_scalers = {}
    for unit in df[config.UNIT_COL].unique():
        scaler_path = scaler_dir / f"{unit}_signal_scaler.pkl"
        if scaler_path.exists():
            signal_scalers[unit] = joblib.load(scaler_path)
        else:
            print(f"⚠  Warning: Scaler not found for unit {unit}")
    
    return {
        'categorical': cat_scaler,
        'signal': signal_scalers
    }

## 7. Window Generation for Training

In [9]:
def create_training_sequences(
    df: pd.DataFrame,
    component: str,
    signal_cols: List[str],
    scalers: Dict[str, Any],
    window_size: int = config.WINDOW_SIZE
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Create sliding window sequences for training.
    
    Uses only Normal data. Scales signals per-unit, then creates overlapping
    windows within each cycle.
    
    Args:
        df: Training DataFrame
        component: Component name
        signal_cols: List of signal columns
        scalers: Dictionary with fitted scalers
        window_size: Number of timesteps per window
    
    Returns:
        Tuple of (signal_sequences, categorical_sequences) as numpy arrays
    """
    # Filter to Normal data
    df_normal = df[df['Label'] == 'Normal'].copy()
    
    signal_sequences = []
    categorical_sequences = []
    
    ohe = scalers['categorical']
    cat_cols_encoded = list(ohe.get_feature_names_out())
    existing_cat_cols = [col for col in config.CAT_COLS if col in df.columns]
    
    for unit in tqdm(df_normal[config.UNIT_COL].unique(), desc="Creating training windows"):
        sc = scalers['signal'][unit]
        unit_data = df_normal[df_normal[config.UNIT_COL] == unit].copy()
        
        # Fill missing and scale
        unit_data[signal_cols] = unit_data[signal_cols].fillna(unit_data[signal_cols].median())
        unit_data[signal_cols] = sc.transform(unit_data[signal_cols])
        
        # Encode categorical
        unit_data[cat_cols_encoded] = ohe.transform(unit_data[existing_cat_cols])
        
        # Create windows per cycle
        for cycle in unit_data['cycle_id'].unique():
            cycle_data = unit_data[unit_data['cycle_id'] == cycle][signal_cols + cat_cols_encoded].values
            
            if cycle_data.shape[0] < window_size * 2:
                continue
            
            # Sliding windows
            for i in range(len(cycle_data) - window_size):
                signal_sequences.append(cycle_data[i:i+window_size, :len(signal_cols)])
                categorical_sequences.append(cycle_data[i:i+window_size, len(signal_cols):])
    
    signal_seq = np.array(signal_sequences)
    cat_seq = np.array(categorical_sequences)
    
    print(f"✓ Created {len(signal_seq):,} training windows")
    print(f"✓ Signal shape: {signal_seq.shape}")
    print(f"✓ Categorical shape: {cat_seq.shape}")
    
    return signal_seq, cat_seq

## 8. Model Architecture

In [10]:
def build_lstm_autoencoder(
    n_signals: int,
    n_cat_features: int,
    sequence_length: int,
    encoder_units_1: int = 16,
    encoder_units_2: int = 8,
    dropout_rate: float = config.DROPOUT_RATE
) -> Model:
    """
    Build LSTM autoencoder for multivariate telemetry.
    
    Architecture:
    - Encoder: 2 LSTM layers with dropout
    - Decoder: 2 LSTM layers with dropout + TimeDistributed Dense output
    - Input: concatenated signals + categorical features
    - Output: reconstructed signals only
    
    Args:
        n_signals: Number of signal features
        n_cat_features: Number of categorical features (after encoding)
        sequence_length: Time steps in sequence
        encoder_units_1: Units in first LSTM layer
        encoder_units_2: Units in second LSTM layer
        dropout_rate: Dropout rate
    
    Returns:
        Compiled Keras Model
    """
    # Encoder
    encoder_inputs = layers.Input(shape=(sequence_length, n_signals + n_cat_features))
    x = layers.LSTM(encoder_units_1, return_sequences=True)(encoder_inputs)
    x = layers.Dropout(dropout_rate)(x)
    encoded = layers.LSTM(encoder_units_2, return_sequences=False)(x)
    
    # Decoder
    x = layers.RepeatVector(sequence_length)(encoded)
    x = layers.LSTM(encoder_units_2, return_sequences=True)(x)
    x = layers.Dropout(dropout_rate)(x)
    x = layers.LSTM(encoder_units_1, return_sequences=True)(x)
    decoded = layers.TimeDistributed(layers.Dense(n_signals))(x)
    
    # Full autoencoder
    autoencoder = Model(encoder_inputs, decoded, name='autoencoder')
    autoencoder.compile(optimizer='adam', loss='mse')
    
    return autoencoder

## 9. Hyperparameter Optimization with Optuna

In [ ]:
def create_optuna_objective(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
    n_signals: int,
    n_cat_features: int,
    sequence_length: int
):
    """
    Create Optuna objective function for hyperparameter optimization.
    
    Args:
        X_train: Training input sequences
        y_train: Training target sequences (signals only)
        X_val: Validation input sequences
        y_val: Validation target sequences
        n_signals: Number of signal features
        n_cat_features: Number of categorical features
        sequence_length: Window size
    
    Returns:
        Objective function for Optuna
    """
    def objective(trial):
        # Suggest hyperparameters
        encoder_units_1 = trial.suggest_int('encoder_units_1', 8, 16, step=4)
        encoder_units_2 = trial.suggest_int('encoder_units_2', 4, 12, step=4)
        dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.4)
        batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])
        learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
        
        # Build model
        model = build_lstm_autoencoder(
            n_signals=n_signals,
            n_cat_features=n_cat_features,
            sequence_length=sequence_length,
            encoder_units_1=encoder_units_1,
            encoder_units_2=encoder_units_2,
            dropout_rate=dropout_rate
        )
        
        # Compile with custom learning rate
        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
            loss='mse'
        )
        
        # Train
        history = model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=2,
            batch_size=batch_size,
            verbose=1,
            callbacks=[
                tf.keras.callbacks.EarlyStopping(
                    patience=3,
                    restore_best_weights=True,
                    monitor='val_loss'
                )
            ]
        )
        
        # Return best validation loss
        return min(history.history['val_loss'])
    
    return objective


def optimize_hyperparameters(
    X_train: np.ndarray,
    y_train: np.ndarray,
    n_signals: int,
    n_cat_features: int,
    sequence_length: int,
    component: str,
    n_trials: int = 20
) -> Dict[str, Any]:
    """
    Run Optuna hyperparameter optimization.
    
    Args:
        X_train: Training input sequences
        y_train: Training target sequences
        n_signals: Number of signal features
        n_cat_features: Number of categorical features
        sequence_length: Window size
        component: Component name
        n_trials: Number of optimization trials
    
    Returns:
        Dictionary with best parameters
    """
    # Split for validation
    val_split = 0.2
    split_idx = int(len(X_train) * (1 - val_split))
    
    X_train_opt = X_train[:split_idx]
    y_train_opt = y_train[:split_idx]
    X_val_opt = X_train[split_idx:]
    y_val_opt = y_train[split_idx:]
    
    print(f"Optimization split: {len(X_train_opt):,} train, {len(X_val_opt):,} val")
    
    # Create study
    study = optuna.create_study(
        direction='minimize',
        study_name=f"{component}_optimization"
    )
    
    # Create objective
    objective = create_optuna_objective(
        X_train_opt, y_train_opt,
        X_val_opt, y_val_opt,
        n_signals, n_cat_features, sequence_length
    )
    
    # Optimize
    print(f"Starting Optuna optimization with {n_trials} trials...")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    
    print(f"✓ Optimization complete")
    print(f"  Best validation loss: {study.best_value:.6f}")
    print(f"  Best parameters: {study.best_params}")
    
    return study.best_params

## 10. MLflow Experiment Tracking

In [12]:
def setup_mlflow_tracking(
    component: str,
    client: str = config.CLIENT
) -> str:
    """
    Setup MLflow tracking with JSON backend.
    
    Args:
        component: Component name
        client: Client identifier
    
    Returns:
        Experiment name
    """
    # Create tracking directory
    tracking_dir = config.BASE_MODEL_PATH / client / component / "mlflow"
    tracking_dir.mkdir(parents=True, exist_ok=True)
    
    # Set tracking URI to file system
    mlflow.set_tracking_uri(f"file:///{tracking_dir.absolute()}")
    
    # Create experiment name with timestamp
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    experiment_name = f"{component}_{config.WINDOW_SIZE}_{timestamp}"
    
    # Set experiment
    mlflow.set_experiment(experiment_name)
    
    print(f"✓ MLflow tracking configured")
    print(f"  Experiment: {experiment_name}")
    print(f"  Location: {tracking_dir}")
    
    return experiment_name


def log_training_run(
    run_name: str,
    params: Dict[str, Any],
    metrics: Dict[str, float],
    model: Model,
    artifacts: Dict[str, Any] = None
):
    """
    Log a training run to MLflow.
    
    Args:
        run_name: Name for the MLflow run
        params: Dictionary of parameters to log
        metrics: Dictionary of metrics to log
        model: Trained Keras model
        artifacts: Optional dictionary of additional artifacts
    """
    with mlflow.start_run(run_name=run_name):
        # Log parameters
        mlflow.log_params(params)
        
        # Log metrics
        mlflow.log_metrics(metrics)
        
        # Log model
        mlflow.keras.log_model(model, "model")
        
        # Log additional artifacts
        if artifacts:
            for name, obj in artifacts.items():
                if isinstance(obj, (dict, list)):
                    # Save as JSON
                    artifact_path = Path(mlflow.get_artifact_uri()) / f"{name}.json"
                    artifact_path.parent.mkdir(parents=True, exist_ok=True)
                    with open(artifact_path, 'w') as f:
                        json.dump(obj, f, indent=2, default=str)
                    mlflow.log_artifact(str(artifact_path))
        
        print(f"✓ Logged run: {run_name}")

## 11. Model Training

In [ ]:
def train_component_model(
    df_train: pd.DataFrame,
    component: str,
    signal_cols: List[str],
    client: str = config.CLIENT,
    optimize: bool = True,
    n_trials: int = 20
) -> Tuple[Model, Dict[str, Any]]:
    """
    Complete training pipeline for a single component.
    
    Includes:
    - Scaler fitting
    - Sequence creation
    - Hyperparameter optimization (optional)
    - Model training
    - MLflow logging
    - Artifact saving
    
    Args:
        df_train: Training DataFrame
        component: Component name
        signal_cols: List of signal columns for this component
        client: Client identifier
        optimize: Whether to run Optuna optimization
        n_trials: Number of Optuna trials
    
    Returns:
        Tuple of (trained_model, training_info)
    """
    print(f"\n{'='*60}")
    print(f"Training model for component: {component}")
    print(f"{'='*60}\n")
    
    # Setup MLflow
    experiment_name = setup_mlflow_tracking(component, client)
    
    # Fit scalers
    print("\n[1/6] Fitting scalers...")
    scalers = fit_scalers_per_unit(df_train, signal_cols, component, client)
    
    # Create training sequences
    print("\n[2/6] Creating training sequences...")
    signal_seq, cat_seq = create_training_sequences(
        df_train, component, signal_cols, scalers
    )
    
    X_train = np.concatenate([signal_seq, cat_seq], axis=-1)
    y_train = signal_seq  # Reconstruct signals only
    
    n_signals = len(signal_cols)
    n_cat_features = cat_seq.shape[-1] if cat_seq.size > 0 else 0
    
    # Hyperparameter optimization
    best_params = {}
    if optimize:
        print("\n[3/6] Optimizing hyperparameters...")
        best_params = optimize_hyperparameters(
            X_train, y_train,
            n_signals, n_cat_features,
            config.WINDOW_SIZE,
            component,
            n_trials=n_trials
        )
    else:
        print("\n[3/6] Skipping optimization, using default parameters")
        best_params = {
            'encoder_units_1': 16,
            'encoder_units_2': 8,
            'dropout_rate': config.DROPOUT_RATE,
            'batch_size': 32,
            'learning_rate': 0.001
        }
    
    # Build model with best parameters
    print("\n[4/6] Building model...")
    model = build_lstm_autoencoder(
        n_signals=n_signals,
        n_cat_features=n_cat_features,
        sequence_length=config.WINDOW_SIZE,
        encoder_units_1=best_params['encoder_units_1'],
        encoder_units_2=best_params['encoder_units_2'],
        dropout_rate=best_params['dropout_rate']
    )
    
    # Compile with optimized learning rate
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=best_params['learning_rate']),
        loss='mse'
    )
    
    print(f"✓ Model created with {model.count_params():,} parameters")
    
    # Train model
    print("\n[5/6] Training model...")
    history = model.fit(
        X_train, y_train,
        epochs=5,
        batch_size=best_params['batch_size'],
        validation_split=0.2,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                patience=config.EARLY_STOPPING_PATIENCE,
                restore_best_weights=True,
                monitor='val_loss'
            )
        ],
        verbose=1
    )
    
    # Save model
    print("\n[6/6] Saving artifacts...")
    model_dir = config.BASE_MODEL_PATH / client / component
    model_dir.mkdir(parents=True, exist_ok=True)
    model_path = model_dir / "model.keras"
    model.save(model_path)
    print(f"✓ Model saved to: {model_path}")
    
    # Prepare training info
    training_info = {
        'component': component,
        'n_signals': n_signals,
        'signal_cols': signal_cols,
        'n_cat_features': n_cat_features,
        'window_size': config.WINDOW_SIZE,
        'n_training_samples': len(X_train),
        'best_params': best_params,
        'final_train_loss': float(history.history['loss'][-1]),
        'final_val_loss': float(history.history['val_loss'][-1]),
        'experiment_name': experiment_name,
        'timestamp': datetime.now().isoformat()
    }
    
    # Save training info
    info_path = model_dir / "training_info.json"
    with open(info_path, 'w') as f:
        json.dump(training_info, f, indent=2)
    print(f"✓ Training info saved to: {info_path}")
    
    # Log to MLflow
    log_training_run(
        run_name=f"{component}_final",
        params={
            **best_params,
            'window_size': config.WINDOW_SIZE,
            'n_signals': n_signals,
            'n_cat_features': n_cat_features
        },
        metrics={
            'final_train_loss': training_info['final_train_loss'],
            'final_val_loss': training_info['final_val_loss']
        },
        model=model,
        artifacts={
            'training_info': training_info,
            'history': history.history,
            'signal_cols': signal_cols
        }
    )
    
    print(f"\n✓ Training complete for {component}")
    print(f"  Final validation loss: {training_info['final_val_loss']:.6f}")
    
    return model, training_info

## 12. Single-Window Inference

In [14]:
def prepare_inference_window(
    df: pd.DataFrame,
    start_time: pd.Timestamp,
    end_time: pd.Timestamp,
    window_size: int = config.WINDOW_SIZE
) -> Tuple[pd.DataFrame, List[str]]:
    """
    Prepare data for inference on a single window.
    
    Creates a complete time index for the window and ensures each unit
    has exactly window_size records.
    
    Args:
        df: Input DataFrame
        start_time: Window start time
        end_time: Window end time
        window_size: Number of timesteps
    
    Returns:
        Tuple of (prepared_df, unit_list)
    """
    out = []
    unit_list = list(df[config.UNIT_COL].unique())
    
    for unit in unit_list:
        unit_data = df[df[config.UNIT_COL] == unit].copy()
        
        # Create complete time index
        full_time_index = pd.date_range(start=start_time, end=end_time, freq=config.FREQ)
        unit_data = unit_data.set_index(config.TIME_COL).reindex(full_time_index).reset_index()
        unit_data = unit_data.rename(columns={'index': config.TIME_COL})
        unit_data[config.UNIT_COL] = unit
        
        # Keep only window_size records
        out.append(unit_data.head(window_size))
    
    return pd.concat(out, ignore_index=True), unit_list


def predict_single_window(
    df: pd.DataFrame,
    start_time: pd.Timestamp,
    end_time: pd.Timestamp,
    component: str,
    signal_cols: List[str],
    client: str = config.CLIENT
) -> pd.DataFrame:
    """
    Run inference on a single window of data.
    
    Args:
        df: Input DataFrame
        start_time: Window start time
        end_time: Window end time
        component: Component name
        signal_cols: List of signal columns
        client: Client identifier
    
    Returns:
        DataFrame with predictions in original scale
    """
    # Load model and scalers
    model_path = config.BASE_MODEL_PATH / client / component / "model.keras"
    model = tf.keras.models.load_model(model_path)
    
    scalers = load_scalers(df, component, client)
    
    # Prepare data
    prepared_df, unit_list = prepare_inference_window(df, start_time, end_time)
    
    # Create sequences (inference mode)
    ohe = scalers['categorical']
    cat_cols_encoded = list(ohe.get_feature_names_out())
    existing_cat_cols = [col for col in config.CAT_COLS if col in df.columns]
    
    signal_sequences = []
    categorical_sequences = []
    
    for unit in unit_list:
        sc = scalers['signal'][unit]
        unit_data = prepared_df[prepared_df[config.UNIT_COL] == unit].copy()
        
        # Scale and fill
        unit_data[signal_cols] = sc.transform(unit_data[signal_cols])
        unit_data[signal_cols] = unit_data[signal_cols].fillna(-10)  # Flag missing values
        
        # Encode categorical
        unit_data[cat_cols_encoded] = ohe.transform(unit_data[existing_cat_cols])
        
        # Create single window
        cycle_data = unit_data[signal_cols + cat_cols_encoded].values
        if cycle_data.shape[0] >= config.WINDOW_SIZE:
            signal_sequences.append(cycle_data[:config.WINDOW_SIZE, :len(signal_cols)])
            categorical_sequences.append(cycle_data[:config.WINDOW_SIZE, len(signal_cols):])
    
    # Predict
    X = np.concatenate([
        np.array(signal_sequences),
        np.array(categorical_sequences)
    ], axis=-1)
    
    predictions_scaled = model.predict(X, verbose=1)
    
    # Inverse transform
    predictions_unscaled = []
    for i, unit in enumerate(unit_list):
        sc = scalers['signal'][unit]
        pred_unscaled = sc.inverse_transform(predictions_scaled[i])
        pred_df = pd.DataFrame(pred_unscaled, columns=signal_cols)
        pred_df[config.UNIT_COL] = unit
        pred_df[config.TIME_COL] = prepared_df[prepared_df[config.UNIT_COL] == unit][config.TIME_COL].values[:len(pred_df)]
        predictions_unscaled.append(pred_df)
    
    return pd.concat(predictions_unscaled, ignore_index=True)

## 13. Multi-Window Inference for Extended Horizons

In [15]:
def predict_over_horizon(
    df: pd.DataFrame,
    component: str,
    signal_cols: List[str],
    client: str = config.CLIENT,
    window_size: int = config.WINDOW_SIZE,
    stride: Optional[int] = None
) -> pd.DataFrame:
    """
    Run inference over an extended time period using sliding windows.
    
    This function handles long inference horizons by breaking them into
    multiple windows and concatenating results.
    
    Args:
        df: Input DataFrame spanning the full inference period
        component: Component name
        signal_cols: List of signal columns
        client: Client identifier
        window_size: Size of each inference window in minutes
        stride: Step size between windows (if None, uses window_size for no overlap)
    
    Returns:
        DataFrame with predictions for the full horizon
    """
    if stride is None:
        stride = window_size  # No overlap by default
    
    print(f"\nRunning multi-window inference for {component}")
    print(f"  Window size: {window_size} minutes")
    print(f"  Stride: {stride} minutes")
    
    # Get time range
    min_time = df[config.TIME_COL].min()
    max_time = df[config.TIME_COL].max()
    total_duration = (max_time - min_time).total_seconds() / 60  # in minutes
    
    print(f"  Time range: {min_time} to {max_time}")
    print(f"  Total duration: {total_duration:.0f} minutes ({total_duration/60:.1f} hours)")
    
    # Generate window boundaries
    window_starts = []
    current_time = min_time
    
    while current_time + pd.Timedelta(minutes=window_size) <= max_time:
        window_starts.append(current_time)
        current_time += pd.Timedelta(minutes=stride)
    
    print(f"  Number of windows: {len(window_starts)}")
    
    # Run inference on each window
    all_predictions = []
    
    for start_time in tqdm(window_starts, desc="Processing windows"):
        end_time = start_time + pd.Timedelta(minutes=window_size)
        
        # Filter data for this window
        window_df = df[
            (df[config.TIME_COL] >= start_time) &
            (df[config.TIME_COL] < end_time)
        ].copy()
        
        if len(window_df) == 0:
            continue
        
        try:
            # Predict
            predictions = predict_single_window(
                window_df, start_time, end_time,
                component, signal_cols, client
            )
            all_predictions.append(predictions)
        except Exception as e:
            print(f"⚠  Warning: Failed to process window starting at {start_time}: {e}")
            continue
    
    # Concatenate all predictions
    if not all_predictions:
        raise ValueError("No predictions were generated")
    
    result = pd.concat(all_predictions, ignore_index=True)
    
    # Remove duplicates if there was overlap
    if stride < window_size:
        result = result.drop_duplicates(subset=[config.UNIT_COL, config.TIME_COL], keep='first')
        result = result.sort_values([config.UNIT_COL, config.TIME_COL]).reset_index(drop=True)
    
    print(f"✓ Generated {len(result):,} predictions across {len(window_starts)} windows")
    
    return result

## 14. Health Index Computation

In [16]:
def compute_error_statistics(
    df_train: pd.DataFrame,
    predictions_train: pd.DataFrame,
    signal_cols: List[str]
) -> Dict[str, Dict[str, float]]:
    """
    Compute error statistics (P50, P95) from training data.
    
    These statistics are used to normalize reconstruction errors during
    health index calculation.
    
    Args:
        df_train: Original training data
        predictions_train: Predicted values on training data
        signal_cols: List of signal columns
    
    Returns:
        Dictionary mapping signal names to their error percentiles
    """
    # Merge original and predicted
    merged = df_train.merge(
        predictions_train,
        on=[config.UNIT_COL, config.TIME_COL],
        suffixes=('_orig', '_pred'),
        how='inner'
    )
    
    error_stats = {}
    
    for col in signal_cols:
        orig_col = f'{col}_orig'
        pred_col = f'{col}_pred'
        
        if orig_col in merged.columns and pred_col in merged.columns:
            # Compute absolute errors
            errors = (merged[orig_col] - merged[pred_col]).abs()
            
            # Calculate percentiles
            p50 = errors.quantile(0.50)
            p95 = errors.quantile(0.95)
            
            error_stats[col] = {
                'p50': float(p50),
                'p95': float(p95)
            }
    
    print(f"✓ Computed error statistics for {len(error_stats)} signals")
    
    return error_stats


def compute_health_index(
    df_original: pd.DataFrame,
    df_predicted: pd.DataFrame,
    signal_cols: List[str],
    error_stats: Dict[str, Dict[str, float]],
    alpha: float = config.HI_ALPHA,
    agg_method: str = config.HI_AGG_METHOD
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Compute health index from reconstruction errors.
    
    Process:
    1. Merge original and predicted data
    2. Compute per-signal absolute reconstruction error
    3. Normalize each error using P50/P95 from error_stats
    4. Aggregate normalized errors across signals at each timestamp
    5. Convert to pointwise health index using exponential decay
    6. Aggregate health index at unit level
    
    Args:
        df_original: Original data
        df_predicted: Predicted/reconstructed data
        signal_cols: List of signal columns
        error_stats: Dictionary with P50/P95 per signal
        alpha: Decay factor for health index (higher = more sensitive)
        agg_method: Temporal aggregation method ('mean' or 'median')
    
    Returns:
        Tuple of (timestamp_level_df, unit_level_df)
    """
    # Merge
    merged = df_original.merge(
        df_predicted,
        on=[config.UNIT_COL, config.TIME_COL],
        suffixes=('_orig', '_pred'),
        how='inner'
    )
    
    merged = merged.sort_values([config.UNIT_COL, config.TIME_COL]).reset_index(drop=True)
    
    norm_error_cols = []
    raw_error_cols = []
    
    eps = 1e-8
    
    # Compute normalized errors per signal
    for col in signal_cols:
        orig_col = f'{col}_orig'
        pred_col = f'{col}_pred'
        err_col = f'{col}_recon_error'
        norm_col = f'{col}_norm_error'
        
        if orig_col not in merged.columns or pred_col not in merged.columns:
            continue
        
        if col not in error_stats:
            print(f"⚠  Warning: No error stats for signal '{col}', skipping")
            continue
        
        p50 = error_stats[col]['p50']
        p95 = error_stats[col]['p95']
        
        if pd.isna(p50) or pd.isna(p95) or p95 <= p50:
            print(f"⚠  Warning: Invalid percentiles for signal '{col}', skipping")
            continue
        
        # Absolute reconstruction error
        merged[err_col] = (merged[orig_col] - merged[pred_col]).abs()
        
        # Normalize relative to healthy distribution
        merged[norm_col] = ((merged[err_col] - p50) / (p95 - p50 + eps)).clip(lower=0)
        
        raw_error_cols.append(err_col)
        norm_error_cols.append(norm_col)
    
    # Aggregate across signals at each timestamp
    merged['reconstruction_error_raw_mean'] = merged[raw_error_cols].mean(axis=1)
    merged['reconstruction_error_norm_mean'] = merged[norm_error_cols].mean(axis=1)
    merged['reconstruction_error_norm_rms'] = np.sqrt((merged[norm_error_cols] ** 2).mean(axis=1))
    
    # Use normalized mean as main anomaly score
    merged['reconstruction_error_score'] = merged['reconstruction_error_norm_mean']
    
    # Convert to health index
    merged['health_index_point'] = np.exp(-alpha * merged['reconstruction_error_score'])
    
    # Aggregate at unit level
    if agg_method == 'mean':
        unit_hi = merged.groupby(config.UNIT_COL, as_index=False).agg(
            health_index=('health_index_point', 'mean'),
            reconstruction_error=('reconstruction_error_score', 'mean'),
            n_records=(config.TIME_COL, 'count'),
            start_time=(config.TIME_COL, 'min'),
            end_time=(config.TIME_COL, 'max')
        )
    elif agg_method == 'median':
        unit_hi = merged.groupby(config.UNIT_COL, as_index=False).agg(
            health_index=('health_index_point', 'median'),
            reconstruction_error=('reconstruction_error_score', 'median'),
            n_records=(config.TIME_COL, 'count'),
            start_time=(config.TIME_COL, 'min'),
            end_time=(config.TIME_COL, 'max')
        )
    else:
        raise ValueError(f"Invalid agg_method: {agg_method}")
    
    print(f"✓ Health index computed for {len(unit_hi)} units")
    print(f"  Mean HI: {unit_hi['health_index'].mean():.3f}")
    print(f"  Std HI: {unit_hi['health_index'].std():.3f}")
    
    return merged, unit_hi

## 15. Artifact Persistence

In [17]:
def save_processed_data(
    df: pd.DataFrame,
    client: str = config.CLIENT
):
    """Save processed training data."""
    output_path = config.BASE_DATA_PATH / f"telemetry/silver/{client}/processed_data.parquet"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(output_path, index=False)
    print(f"✓ Saved processed data to: {output_path}")


def save_inferences(
    df: pd.DataFrame,
    component: str,
    client: str = config.CLIENT
):
    """Save model predictions."""
    output_path = config.BASE_DATA_PATH / f"telemetry/golden/{client}/{component}_inferences.parquet"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(output_path, index=False)
    print(f"✓ Saved inferences to: {output_path}")


def save_health_index(
    df_timestamp: pd.DataFrame,
    df_unit: pd.DataFrame,
    component: str,
    client: str = config.CLIENT
):
    """Save health index results."""
    # Timestamp-level
    timestamp_path = config.BASE_DATA_PATH / f"telemetry/golden/{client}/{component}_health_index_timestamp.parquet"
    timestamp_path.parent.mkdir(parents=True, exist_ok=True)
    df_timestamp.to_parquet(timestamp_path, index=False)
    print(f"✓ Saved timestamp-level health index to: {timestamp_path}")
    
    # Unit-level
    unit_path = config.BASE_DATA_PATH / f"telemetry/golden/{client}/{component}_health_index_unit.parquet"
    df_unit.to_parquet(unit_path, index=False)
    print(f"✓ Saved unit-level health index to: {unit_path}")


def save_error_statistics(
    error_stats: Dict[str, Dict[str, float]],
    component: str,
    client: str = config.CLIENT
):
    """Save error statistics for future inference."""
    output_path = config.BASE_MODEL_PATH / client / component / "error_statistics.json"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    with open(output_path, 'w') as f:
        json.dump(error_stats, f, indent=2)
    
    print(f"✓ Saved error statistics to: {output_path}")


def load_error_statistics(
    component: str,
    client: str = config.CLIENT
) -> Dict[str, Dict[str, float]]:
    """Load error statistics from disk."""
    path = config.BASE_MODEL_PATH / client / component / "error_statistics.json"
    
    with open(path, 'r') as f:
        return json.load(f)

## 16. Single-Component Pipeline Execution

In [18]:
def run_component_pipeline(
    component: str,
    signal_cols: List[str],
    df_train: pd.DataFrame,
    df_test: pd.DataFrame,
    client: str = config.CLIENT,
    optimize: bool = True,
    n_trials: int = 20
) -> Dict[str, Any]:
    """
    Execute complete pipeline for a single component.
    
    Steps:
    1. Train model with optimization
    2. Compute error statistics on training data
    3. Run multi-window inference on test data
    4. Compute health index
    5. Save all artifacts
    
    Args:
        component: Component name
        signal_cols: List of signal columns for this component
        df_train: Training DataFrame
        df_test: Test DataFrame
        client: Client identifier
        optimize: Whether to run hyperparameter optimization
        n_trials: Number of Optuna trials
    
    Returns:
        Dictionary with pipeline results and metadata
    """
    print(f"\n{'#'*80}")
    print(f"# COMPONENT PIPELINE: {component}")
    print(f"{'#'*80}\n")
    
    results = {
        'component': component,
        'status': 'started',
        'start_time': datetime.now().isoformat()
    }
    
    try:
        # ========== TRAINING ==========
        print("\n[PHASE 1: TRAINING]")
        model, training_info = train_component_model(
            df_train=df_train,
            component=component,
            signal_cols=signal_cols,
            client=client,
            optimize=optimize,
            n_trials=n_trials
        )
        
        results['training_info'] = training_info
        
        # ========== ERROR STATISTICS ==========
        print("\n[PHASE 2: ERROR STATISTICS]")
        print("Computing error statistics on training data...")
        
        # Run inference on a sample of training data to compute error stats
        train_sample = df_train.sample(min(10000, len(df_train)), random_state=42)
        train_predictions = predict_over_horizon(
            df=train_sample,
            component=component,
            signal_cols=signal_cols,
            client=client
        )
        
        error_stats = compute_error_statistics(
            df_train=train_sample,
            predictions_train=train_predictions,
            signal_cols=signal_cols
        )
        
        save_error_statistics(error_stats, component, client)
        results['error_stats'] = error_stats
        
        # ========== INFERENCE ==========
        print("\n[PHASE 3: INFERENCE]")
        test_predictions = predict_over_horizon(
            df=df_test,
            component=component,
            signal_cols=signal_cols,
            client=client
        )
        
        save_inferences(test_predictions, component, client)
        results['n_predictions'] = len(test_predictions)
        
        # ========== HEALTH INDEX ==========
        print("\n[PHASE 4: HEALTH INDEX]")
        hi_timestamp, hi_unit = compute_health_index(
            df_original=df_test,
            df_predicted=test_predictions,
            signal_cols=signal_cols,
            error_stats=error_stats
        )
        
        save_health_index(hi_timestamp, hi_unit, component, client)
        results['health_index_summary'] = {
            'mean': float(hi_unit['health_index'].mean()),
            'std': float(hi_unit['health_index'].std()),
            'min': float(hi_unit['health_index'].min()),
            'max': float(hi_unit['health_index'].max())
        }
        
        # ========== COMPLETION ==========
        results['status'] = 'completed'
        results['end_time'] = datetime.now().isoformat()
        
        print(f"\n{'='*60}")
        print(f"✓ Component pipeline completed: {component}")
        print(f"{'='*60}")
        
    except Exception as e:
        results['status'] = 'failed'
        results['error'] = str(e)
        results['end_time'] = datetime.now().isoformat()
        print(f"\nâœ— Component pipeline failed: {component}")
        print(f"  Error: {e}")
        raise
    
    return results

## 17. Multi-Component Pipeline Execution

In [19]:
def run_all_components_pipeline(
    component_mapping: Dict,
    df_train: pd.DataFrame,
    df_test: pd.DataFrame,
    client: str = config.CLIENT,
    optimize: bool = True,
    n_trials: int = 20,
    components_subset: Optional[List[str]] = None
) -> Dict[str, Any]:
    """
    Execute pipeline for all components.
    
    Args:
        component_mapping: Component configuration dictionary
        df_train: Training DataFrame
        df_test: Test DataFrame
        client: Client identifier
        optimize: Whether to run hyperparameter optimization
        n_trials: Number of Optuna trials per component
        components_subset: Optional list of specific components to process
    
    Returns:
        Dictionary with results for all components
    """
    print(f"\n{'#'*80}")
    print(f"# MULTI-COMPONENT PIPELINE")
    print(f"{'#'*80}\n")
    
    components_to_process = components_subset or list(component_mapping['components'].keys())
    
    print(f"Components to process: {components_to_process}")
    print(f"Optimization: {'Enabled' if optimize else 'Disabled'}")
    print(f"Trials per component: {n_trials}\n")
    
    all_results = {
        'client': client,
        'start_time': datetime.now().isoformat(),
        'components': {},
        'summary': {}
    }
    
    successful = 0
    failed = 0
    
    for component in components_to_process:
        print(f"\n{'='*80}")
        print(f"Processing component {successful + failed + 1}/{len(components_to_process)}: {component}")
        print(f"{'='*80}")
        
        try:
            signal_cols = component_mapping['components'][component]['signals']
            
            # Filter to existing columns
            signal_cols = [col for col in signal_cols if col in df_train.columns]
            
            if not signal_cols:
                print(f"⚠  Warning: No valid signal columns for {component}, skipping")
                continue
            
            # Run component pipeline
            component_results = run_component_pipeline(
                component=component,
                signal_cols=signal_cols,
                df_train=df_train,
                df_test=df_test,
                client=client,
                optimize=optimize,
                n_trials=n_trials
            )
            
            all_results['components'][component] = component_results
            successful += 1
            
        except Exception as e:
            print(f"\nâœ— Failed to process component: {component}")
            print(f"  Error: {e}")
            all_results['components'][component] = {
                'status': 'failed',
                'error': str(e)
            }
            failed += 1
            continue
    
    all_results['end_time'] = datetime.now().isoformat()
    all_results['summary'] = {
        'total_components': len(components_to_process),
        'successful': successful,
        'failed': failed,
        'success_rate': successful / len(components_to_process) if components_to_process else 0
    }
    
    # Save overall results
    results_path = config.BASE_MODEL_PATH / client / "pipeline_results.json"
    results_path.parent.mkdir(parents=True, exist_ok=True)
    with open(results_path, 'w') as f:
        json.dump(all_results, f, indent=2, default=str)
    
    print(f"\n{'#'*80}")
    print(f"# PIPELINE COMPLETE")
    print(f"{'#'*80}")
    print(f"\nSuccessful: {successful}/{len(components_to_process)}")
    print(f"Failed: {failed}/{len(components_to_process)}")
    print(f"\nResults saved to: {results_path}")
    
    return all_results

## 18. Main Pipeline Execution

In [20]:
# Load data
print("="*60)
print("LOADING DATA")
print("="*60)

df_raw = load_telemetry_data(client=config.CLIENT)
component_mapping = load_component_mapping(client=config.CLIENT)

LOADING DATA
Loading data from: ..\data\telemetry\silver\cda\Telemetry_Wide_With_States
✓ Loaded 4,183,578 rows (4.18M)
✓ Units: 11
✓ Time range: 2025-01-01 00:00:00 to 2026-01-31 00:00:00
✓ Loaded component mapping with 4 components


In [21]:
# Clean data
print("\n" + "="*60)
print("CLEANING DATA")
print("="*60)

margins = config.get_signal_margins()
df_cleaned = clean_outliers(df_raw, margins)

num_cols = [col for col in margins.keys() if col in df_cleaned.columns]
df_cleaned = drop_incomplete_rows(df_cleaned, num_cols)


CLEANING DATA
  GPSLat: removed 10,082 outliers (0.2%)
  GPSLon: removed 10,082 outliers (0.2%)
  GPSElevation: removed 442 outliers (0.0%)
  GroundSpd: removed 1,730 outliers (0.0%)
  EngSpd: removed 50 outliers (0.0%)
  EngCoolTemp: removed 6,369 outliers (0.2%)
  RAftrclrTemp: removed 1,583 outliers (0.0%)
  EngOilPres: removed 12,586 outliers (0.3%)
  CnkcasePres: removed 282 outliers (0.0%)
  RtLtExhTemp: removed 527,526 outliers (15.7%)
  RtExhTemp: removed 166,470 outliers (4.0%)
  LtExhTemp: removed 140,629 outliers (3.4%)
  DiffLubePres: removed 3,373 outliers (0.1%)
  DiffTemp: removed 148 outliers (0.0%)
  TCOutTemp: removed 39,801 outliers (1.0%)
  RtRBrkTemp: removed 3,668 outliers (0.1%)
  RtFBrkTemp: removed 61,225 outliers (1.6%)
  LtRBrkTemp: removed 4,188 outliers (0.1%)
  LtFBrkTemp: removed 4,712 outliers (0.1%)
✓ Dropped 54,394 incomplete rows (1.3%)
✓ Remaining: 4,129,184 rows


In [22]:
# Process cycles
print("\n" + "="*60)
print("PROCESSING CYCLES")
print("="*60)

df_cycles = create_cycles(df_cleaned)
df_interpolated = process_all_cycles(df_cycles, num_cols)


PROCESSING CYCLES
✓ Created 18,483 cycles across 11 units


Processing cycles:   0%|          | 0/18483 [00:00<?, ?it/s]

✓ Processed 7,035 valid cycles
✓ Total rows after interpolation: 3,120,241


In [23]:
# Label data
print("\n" + "="*60)
print("LABELING DATA")
print("="*60)

df_labeled = label_cycles_by_percentiles(df_interpolated, num_cols)


LABELING DATA
✓ Labeling complete:
  Normal cycles: 993
  Anomalous cycles: 6,042


In [24]:
# Split train/test
print("\n" + "="*60)
print("SPLITTING TRAIN/TEST")
print("="*60)

test_start_date = df_labeled[config.TIME_COL].max() - pd.Timedelta(weeks=config.WEEKS_TO_TEST)

df_train = df_labeled[df_labeled[config.TIME_COL] < test_start_date].copy()
df_test = df_labeled[df_labeled[config.TIME_COL] >= test_start_date].copy()

print(f"✓ Training data: {len(df_train):,} rows")
print(f"  Time range: {df_train[config.TIME_COL].min()} to {df_train[config.TIME_COL].max()}")
print(f"✓ Test data: {len(df_test):,} rows")
print(f"  Time range: {df_test[config.TIME_COL].min()} to {df_test[config.TIME_COL].max()}")

# Save processed data
save_processed_data(df_labeled, client=config.CLIENT)


SPLITTING TRAIN/TEST
✓ Training data: 2,711,645 rows
  Time range: 2025-01-01 00:02:00 to 2025-12-05 23:59:00
✓ Test data: 408,596 rows
  Time range: 2025-12-06 00:00:00 to 2026-01-31 00:00:00
✓ Saved processed data to: ..\data\telemetry\silver\cda\processed_data.parquet


### Single Component Example

Run pipeline for one component as a validation test:

In [25]:
# Example: Run pipeline for a single component

RUN_COMPONENT_PIPELINE = False  # Set to True to run the example pipeline for a single component
component_example = "Tren de Fuerza"  # Change this to test different components

if RUN_COMPONENT_PIPELINE:
    if component_example in component_mapping['components']:
        signal_cols = component_mapping['components'][component_example]['signals']
        signal_cols = [col for col in signal_cols if col in df_train.columns]
        
        example_results = run_component_pipeline(
            component=component_example,
            signal_cols=signal_cols,
            df_train=df_train,
            df_test=df_test,
            client=config.CLIENT,
            optimize=True,
            n_trials=10  # Use fewer trials for testing
        )
        
        print(f"\n{'='*60}")
        print("EXAMPLE RESULTS")
        print(f"{'='*60}")
        print(json.dumps(example_results, indent=2, default=str))
    else:
        print(f"Component '{component_example}' not found in mapping")
else:
    print("Skipping component pipeline execution. Set RUN_COMPONENT_PIPELINE = True to run.")

Skipping component pipeline execution. Set RUN_COMPONENT_PIPELINE = True to run.


### Full Multi-Component Execution

Run the complete pipeline across all components:

In [ ]:
# Run full multi-component pipeline
# WARNING: This will take significant time depending on data size and number of components

RUN_FULL_PIPELINE = True  # Set to True to execute

if RUN_FULL_PIPELINE:
    all_results = run_all_components_pipeline(
        component_mapping=component_mapping,
        df_train=df_train,
        df_test=df_test,
        client=config.CLIENT,
        optimize=True,
        n_trials=2,
        components_subset=None  # Set to list of component names to process specific ones
    )
    
    print(f"\n{'='*60}")
    print("FINAL SUMMARY")
    print(f"{'='*60}")
    print(json.dumps(all_results['summary'], indent=2))
else:
    print("Skipping full pipeline execution. Set RUN_FULL_PIPELINE = True to run.")

2026/03/18 17:47:07 INFO mlflow.tracking.fluent: Experiment with name 'Motor_60_20260318_174707' does not exist. Creating a new experiment.



################################################################################
# MULTI-COMPONENT PIPELINE
################################################################################

Components to process: ['Motor', 'Tren de fuerza', 'Frenos', 'Direccion']
Optimization: Enabled
Trials per component: 2


Processing component 1/4: Motor

################################################################################
# COMPONENT PIPELINE: Motor
################################################################################


[PHASE 1: TRAINING]

Training model for component: Motor

✓ MLflow tracking configured
  Experiment: Motor_60_20260318_174707
  Location: ..\models\cda\Motor\mlflow

[1/6] Fitting scalers...
✓ Fitted scalers for 9 units
✓ Categorical features: 9
✓ Saved to: ..\models\cda\Motor\scalers

[2/6] Creating training sequences...


Creating training windows:   0%|          | 0/9 [00:00<?, ?it/s]

✓ Created 348,631 training windows
✓ Signal shape: (348631, 60, 7)
✓ Categorical shape: (348631, 60, 9)


[I 2026-03-18 17:47:18,350] A new study created in memory with name: Motor_optimization



[3/6] Optimizing hyperparameters...
Optimization split: 278,904 train, 69,727 val
Starting Optuna optimization with 2 trials...
Epoch 1/20
15057/17432 ━━━━━━━━━━━━━━━━━━━━ 1:12 31ms/step - loss: 0.2692

## 19. Results Visualization

Load and visualize health index results:

In [ ]:
def plot_health_index_summary(component: str, client: str = config.CLIENT):
    """Plot health index summary for a component."""
    try:
        # Load unit-level health index
        hi_path = config.BASE_DATA_PATH / f"telemetry/golden/{client}/{component}_health_index_unit.parquet"
        hi_df = pd.read_parquet(hi_path)
        
        # Create figure
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        fig.suptitle(f'Health Index Summary: {component}', fontsize=16, fontweight='bold')
        
        # 1. Health index distribution
        axes[0, 0].hist(hi_df['health_index'], bins=30, edgecolor='black', alpha=0.7)
        axes[0, 0].axvline(hi_df['health_index'].mean(), color='red', linestyle='--', label=f'Mean: {hi_df["health_index"].mean():.3f}')
        axes[0, 0].set_xlabel('Health Index')
        axes[0, 0].set_ylabel('Frequency')
        axes[0, 0].set_title('Health Index Distribution')
        axes[0, 0].legend()
        axes[0, 0].grid(alpha=0.3)
        
        # 2. Health index by unit
        hi_df_sorted = hi_df.sort_values('health_index')
        axes[0, 1].barh(range(len(hi_df_sorted)), hi_df_sorted['health_index'])
        axes[0, 1].set_yticks(range(len(hi_df_sorted)))
        axes[0, 1].set_yticklabels(hi_df_sorted[config.UNIT_COL], fontsize=8)
        axes[0, 1].set_xlabel('Health Index')
        axes[0, 1].set_title('Health Index by Unit')
        axes[0, 1].grid(alpha=0.3, axis='x')
        
        # 3. Reconstruction error vs health index
        axes[1, 0].scatter(hi_df['reconstruction_error'], hi_df['health_index'], alpha=0.6)
        axes[1, 0].set_xlabel('Reconstruction Error')
        axes[1, 0].set_ylabel('Health Index')
        axes[1, 0].set_title('Reconstruction Error vs Health Index')
        axes[1, 0].grid(alpha=0.3)
        
        # 4. Summary statistics
        axes[1, 1].axis('off')
        summary_text = f"""
        Summary Statistics
        {'='*40}
        
        Number of Units: {len(hi_df)}
        
        Health Index:
          Mean:   {hi_df['health_index'].mean():.4f}
          Median: {hi_df['health_index'].median():.4f}
          Std:    {hi_df['health_index'].std():.4f}
          Min:    {hi_df['health_index'].min():.4f}
          Max:    {hi_df['health_index'].max():.4f}
        
        Reconstruction Error:
          Mean:   {hi_df['reconstruction_error'].mean():.4f}
          Median: {hi_df['reconstruction_error'].median():.4f}
          Std:    {hi_df['reconstruction_error'].std():.4f}
        
        Total Records: {hi_df['n_records'].sum():,}
        """
        axes[1, 1].text(0.1, 0.5, summary_text, fontsize=11, family='monospace',
                        verticalalignment='center')
        
        plt.tight_layout()
        plt.show()
        
        return hi_df
        
    except FileNotFoundError:
        print(f"Health index file not found for component: {component}")
        print("Run the pipeline first to generate results.")
        return None


# Example: Visualize results for a component
if component_example in component_mapping['components']:
    hi_results = plot_health_index_summary(component_example, config.CLIENT)

## 20. Pipeline Complete

This notebook implements a modular, scalable pipeline for telemetry-based health index modeling with:

✓ **Modular design** - Clean separation of concerns with single-responsibility functions  
✓ **Hyperparameter optimization** - Optuna integration for automated tuning  
✓ **Experiment tracking** - MLflow logging with JSON backend  
✓ **Multi-window inference** - Support for extended time horizons  
✓ **Multi-component execution** - Batch processing across all components  
✓ **Proper artifact management** - Organized storage of models, scalers, and results  

**Next steps:**
1. Adjust configuration parameters in `PipelineConfig` as needed
2. Run single-component pipeline to validate
3. Execute full multi-component pipeline
4. Analyze health index results
5. Iterate and refine based on domain knowledge